# MICrONS Option 4 — Time-resolved trajectories of Clip subcategories

Replicates the conceptual idea of Option 3 (per-area, per-stim trial-averaged trajectories with the time axis preserved) but **inside the Clip stimulus class**, using the three content categories MICrONS already labels (`Cinematic` / `sports1m` / `Rendered`).

Design and analysis details: `docs/specs/2026-05-07-option4-clip-categories-design.md`.

## How to use this notebook

Pipeline is **session-swappable**: change `SESSION` in the configuration block below and restart the kernel.

## Part 0 — Setup

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import microns_eda
# Used in Part 2 below.
import option2_pca_utils
import option3_trajectories_utils as traj_utils
import option4_clip_utils as o4

In [ ]:
# === Configuration block — every tunable lives here. ===
SESSION = "7_5"

# Preprocessing (inherited from Option 2 / Option 3).
N_FRAMES_TRUNCATE = 75
TREADMILL_OUTLIER_THRESHOLD = 1.0

# PCA / metrics.
N_COMPONENTS = 10
N_PCS_FOR_DISTANCE = 3

# Bootstrap / null.
N_BOOT = 1000
N_SHUFFLES = 100

# Population-matching control.
N_POPULATION_SUBSAMPLES = 20

# Catalog (Part 1).
N_PER_CATEGORY_THUMBNAILS = 8
N_FRAMES_PER_STRIP = 6
THUMBNAIL_GRID_NCOLS = 16

# Reproducibility.
RANDOM_SEED = 42

# Paths.
DATADIR = Path(os.environ.get("MICRONS_DATADIR", "../neuroscience"))
FIGURES_DIR = Path(f"figures/option4/{SESSION}")
RESULTS_DIR = Path(f"results/option4/{SESSION}")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# 7.5 Hz sampling → ~133 ms/frame, ~10 s total trial window.
def frame_to_ms(f):
    return f * 1000.0 / 7.5

np.random.seed(RANDOM_SEED)

print(f"SESSION       = {SESSION}")
print(f"DATADIR       = {DATADIR}")
print(f"FIGURES_DIR   = {FIGURES_DIR}")
print(f"RESULTS_DIR   = {RESULTS_DIR}")

In [ ]:
# Ensure the H5 lives where MicronsFunctionalReader expects it.
expected = DATADIR / "functional" / "microns_functional.h5"
if not expected.exists():
    candidates = [
        DATADIR / "microns.h5",
        DATADIR / "microns_functional.h5",
    ]
    source = next((c for c in candidates if c.exists()), None)
    if source is None:
        raise FileNotFoundError(
            f"H5 not found at {expected} or at any of {candidates}. "
            f"Set MICRONS_DATADIR to the directory containing your H5 file."
        )
    expected.parent.mkdir(parents=True, exist_ok=True)
    expected.symlink_to(source.resolve())
    print(f"Created symlink: {expected} -> {source.resolve()}")
else:
    print(f"H5 found at expected path: {expected}")

In [ ]:
# === Sanity check + per-trial labels ===
reader = microns_eda.open_dataset(DATADIR)
sessions = microns_eda.list_sessions(DATADIR)
assert SESSION in sessions, f"session {SESSION!r} not in {sessions}"

meta = microns_eda.get_session_meta(DATADIR, SESSION)
print(f"  n_neurons = {meta['n_neurons']}")
print(f"  n_trials  = {meta['n_trials']}")

# Build per-trial stim type and per-trial Clip category.
stim_per_trial, category_per_trial = o4.category_labels_for_session(
    reader, DATADIR, SESSION,
)

# Per-trial running QC.
n_trials_total = meta["n_trials"]
per_trial_tread_means = np.empty(n_trials_total, dtype=np.float64)
for i in range(n_trials_total):
    trial = microns_eda.load_trial(reader, DATADIR, SESSION, i)
    per_trial_tread_means[i] = np.nanmean(trial["treadmill"])
clean_trial_indices, running_mask = microns_eda.compute_clean_trial_indices(
    per_trial_tread_means, threshold=TREADMILL_OUTLIER_THRESHOLD,
)

# Restrict to Clip trials only — Option 4 is purely within-Clip.
clip_trial_indices, clip_category_labels = o4.restrict_to_clip(
    clean_trial_indices, stim_per_trial, category_per_trial,
)

category_post_qc_counts = dict(pd.Series(clip_category_labels).value_counts())
print(f"\nclean trials (post running QC): {len(clean_trial_indices)}")
print(f"  of which Clip-class           : {len(clip_trial_indices)}")
print(f"  per category (post-QC):")
for cat in o4.CATEGORY_ORDER:
    print(f"    {cat:10s} {category_post_qc_counts.get(cat, 0)}")

# Sanity asserts: the categories we expect are exactly the ones we get.
assert set(clip_category_labels) <= set(o4.CATEGORY_ORDER), (
    f"unexpected categories: {set(clip_category_labels) - set(o4.CATEGORY_ORDER)}"
)
for cat in o4.CATEGORY_ORDER:
    assert category_post_qc_counts.get(cat, 0) > 0, f"no trials in category {cat}"

## Part 1 — Catalog the Clip class

Clip is heterogeneous: each trial shows a different short movie excerpt, and MICrONS labels each excerpt with one of three content categories. Part 1 builds the inventory and visualises what each category contains, so that Part 2's category-resolved trajectory analysis is grounded in interpretable labels.

### 1.1 Inventory + balance table

In [ ]:
# Decoded hashes for the post-QC Clip trials.
raw_hashes = meta["condition_hashes"]
hashes_decoded_full = np.array([
    h.decode("utf-8", "replace") if isinstance(h, bytes) else str(h)
    for h in raw_hashes
], dtype=object)
clip_hashes_per_trial = hashes_decoded_full[clip_trial_indices]

h5_path = DATADIR / "functional" / "microns_functional.h5"
hash_to_category = o4.build_hash_to_category(h5_path)

inventory = o4.build_clip_inventory(reader, clip_hashes_per_trial, hash_to_category, h5_path)

csv_path = RESULTS_DIR / "clip_inventory.csv"
inventory.to_csv(csv_path, index=False)
print(f"wrote {csv_path}  ({len(inventory)} unique clips)")
inventory.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 2.6))
o4.plot_balance_table(inventory, category_post_qc_counts, ax=ax)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "1_1_balance_table.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
o4.plot_trial_count_histogram(inventory, ax=ax)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "1_1_trial_count_histogram.png", dpi=150)
plt.show()

### 1.2 Category exemplars

In [ ]:
fig = o4.plot_category_exemplars(reader, inventory, n_per_category=N_PER_CATEGORY_THUMBNAILS)
fig.savefig(FIGURES_DIR / "1_2_category_exemplars.png", dpi=150, bbox_inches="tight")
plt.show()

### 1.3 Frame-strip preview (within-clip evolution)

In [ ]:
fig = o4.plot_frame_strip_per_category(reader, inventory, n_frames=N_FRAMES_PER_STRIP)
fig.savefig(FIGURES_DIR / "1_3_frame_strips.png", dpi=150, bbox_inches="tight")
plt.show()

### 1.4 Big-picture catalog

One mid-frame per unique Clip movie, sorted by category (Cinematic block, sports1m block, Rendered block). Panel titles colored by category. Saved at `dpi=100` to keep the file size manageable.

In [ ]:
fig = o4.plot_full_thumbnail_grid(reader, inventory, ncols=THUMBNAIL_GRID_NCOLS)
if fig is None:
    print(f"Inventory has > 400 unique clips; skipped the big-picture grid.")
else:
    fig.savefig(FIGURES_DIR / "1_4_full_thumbnail_grid.png", dpi=100, bbox_inches="tight")
    plt.show()

## Part 2 — Per-category trajectories

Mirrors Option 3 part-for-part with the labels swapped from `Clip / Monet2 / Trippy` to `Cinematic / sports1m / Rendered`. Pipeline is reused unchanged from `option3_trajectories_utils`; only the palette and stim_order kwargs change.

### 2.0 Build trajectories

Per-category, per-area trial-averaged trajectories with the time axis preserved. Uses the existing Option 2 preprocessing (detrend → z-score) and the existing Option 3 trajectory builder.

In [ ]:
print("loading + preprocessing responses...")
responses, trial_boundaries, _ = microns_eda.load_session_responses(
    reader, DATADIR, SESSION,
)
responses_pp = option2_pca_utils.preprocess_responses(responses, apply_log=False)
print(f"  preprocessed shape = {responses_pp.shape}")

brain_areas_str = np.array([
    a.decode("utf-8", "replace") if isinstance(a, bytes) else str(a)
    for a in meta["brain_areas"]
], dtype=object)

trajectories = traj_utils.build_stim_trajectories(
    responses_pp, trial_boundaries,
    clip_trial_indices,
    brain_areas_str,
    clip_category_labels,
    n_frames=N_FRAMES_TRUNCATE,
)

for area, per_cat in trajectories.items():
    shapes = {c: t.shape for c, t in per_cat.items()}
    print(f"  {area}: {shapes}")

# Sanity asserts.
for area in ["V1", "AL", "LM", "RL"]:
    assert area in trajectories
    for cat in o4.CATEGORY_ORDER:
        assert cat in trajectories[area], f"no trajectory for {area}/{cat}"
        t = trajectories[area][cat]
        assert t.shape[0] == N_FRAMES_TRUNCATE, f"frame count mismatch {area}/{cat}: {t.shape}"
        assert not np.any(np.isnan(t)), f"NaN in {area}/{cat}"

traj_path = RESULTS_DIR / "trajectories.npz"
traj_utils.save_trajectories(trajectories, traj_path)
print(f"\nwrote {traj_path}")

### 2.1 Fit per-area PCA on stacked-category matrix

Stack the three trajectories vertically per area (3 × 75 = 225 frames × n_neurons), then fit a per-area PCA. PC1 typically captures stimulus-common temporal dynamics (rise at onset, settle later); the category-identity signal lives in PC2-PC3 in the visualisations. Reuses the Option 3 helper unchanged.

In [ ]:
pca_per_area = traj_utils.fit_trajectory_pca(
    trajectories, n_components=N_COMPONENTS, random_state=RANDOM_SEED,
)
for area in ["V1", "AL", "LM", "RL"]:
    cum = np.cumsum(pca_per_area[area]["pca"].explained_variance_ratio_)
    print(f"{area:5s} top-3 cum.var = {cum[2]:.3f},  top-10 cum.var = {cum[9]:.3f}")

### 2.2 V1 deep-dive

2-D and 3-D scatters of the trajectories, full-feature and top-3-PC distance time courses, bootstrap envelopes (B=1000), shuffle null (n=100), and onset latency per pair.

In [ ]:
v1_pca = pca_per_area["V1"]["pca"]
v1_stim_pcs = pca_per_area["V1"]["stim_pcs"]

# 2-D scatter.
fig, ax = plt.subplots(figsize=(8, 6))
traj_utils.plot_trajectory_2d(
    v1_stim_pcs, "V1", v1_pca,
    palette=o4.CATEGORY_COLORS, stim_order=o4.CATEGORY_ORDER,
    ax=ax,
)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "2_2_v1_trajectory_2d.png", dpi=150)
plt.show()

# 3-D Plotly scatter.
fig3d = traj_utils.plot_trajectory_3d_plotly(
    v1_stim_pcs, "V1", v1_pca,
    palette=o4.CATEGORY_COLORS, stim_order=o4.CATEGORY_ORDER,
)
fig3d.write_html(str(FIGURES_DIR / "2_2_v1_trajectory_3d.html"))
print("saved 3D Plotly to 2_2_v1_trajectory_3d.html")

In [ ]:
# Build the V1 trial × time × neuron tensor (used by bootstrap + null).
# Same helper Option 3's notebook defines locally — kept inline here so this
# notebook stays self-contained and option3_trajectories_utils.py is unchanged.
def _build_trial_tensor(responses_pp, trial_boundaries, trial_indices, cols, n_frames):
    out = np.empty((len(trial_indices), n_frames, len(cols)), dtype=np.float64)
    for out_idx, t_idx in enumerate(trial_indices):
        start = int(trial_boundaries[t_idx])
        out[out_idx] = responses_pp[cols, start:start + n_frames].T
    return out

v1_tensor_cols = np.where(brain_areas_str == "V1")[0]
v1_tensor = _build_trial_tensor(
    responses_pp, trial_boundaries, clip_trial_indices, v1_tensor_cols,
    N_FRAMES_TRUNCATE,
)
print(f"V1 tensor shape: {v1_tensor.shape}")

v1_d_full = traj_utils.pairwise_trajectory_distance(
    trajectories["V1"], metric="full",
)
v1_d_top3 = traj_utils.pairwise_trajectory_distance(
    trajectories["V1"], metric="top_pcs",
    pca=v1_pca, n_pcs=N_PCS_FOR_DISTANCE,
)

print(f"bootstrap envelopes (B={N_BOOT})...")
v1_env_full = traj_utils.bootstrap_distance_envelope(
    v1_tensor, clip_category_labels, metric="full",
    n_boot=N_BOOT, seed=RANDOM_SEED + 1,
)
v1_env_top3 = traj_utils.bootstrap_distance_envelope(
    v1_tensor, clip_category_labels, metric="top_pcs",
    pca=v1_pca, n_pcs=N_PCS_FOR_DISTANCE,
    n_boot=N_BOOT, seed=RANDOM_SEED + 2,
)
print(f"shuffle null (n={N_SHUFFLES})...")
v1_null_full = traj_utils.shuffle_null_max_distance(
    v1_tensor, clip_category_labels, metric="full",
    n_shuffles=N_SHUFFLES, seed=RANDOM_SEED + 3,
)
v1_null_top3 = traj_utils.shuffle_null_max_distance(
    v1_tensor, clip_category_labels, metric="top_pcs",
    pca=v1_pca, n_pcs=N_PCS_FOR_DISTANCE,
    n_shuffles=N_SHUFFLES, seed=RANDOM_SEED + 4,
)

In [ ]:
# Two-panel V1 figure: full features + top-3 PC.
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
traj_utils.plot_pairwise_distance_time_course(
    v1_d_full, v1_env_full, "V1", "Euclidean (full features)",
    null_p95={p: float(np.percentile(v1_null_full[p], 95)) for p in v1_null_full},
    pair_palette=o4.category_pair_palette(),
    pair_labels=o4.category_pair_labels(),
    ax=axes[0],
)
traj_utils.plot_pairwise_distance_time_course(
    v1_d_top3, v1_env_top3, "V1", "Euclidean (top-3 PCs)",
    null_p95={p: float(np.percentile(v1_null_top3[p], 95)) for p in v1_null_top3},
    pair_palette=o4.category_pair_palette(),
    pair_labels=o4.category_pair_labels(),
    ax=axes[1],
)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "2_2_v1_distances.png", dpi=150)
plt.show()

### 2.3 Repeat on AL, LM, RL

In [ ]:
all_area_results = {
    "V1": {
        "tensor": v1_tensor,
        "trajectories": trajectories["V1"],
        "pca": v1_pca,
        "stim_pcs": v1_stim_pcs,
        "distances_full": v1_d_full,
        "distances_top3": v1_d_top3,
        "envelopes_full": v1_env_full,
        "envelopes_top3": v1_env_top3,
        "null_full": v1_null_full,
        "null_top3": v1_null_top3,
        "n_neurons": v1_tensor.shape[2],
    },
}

for area in ["AL", "LM", "RL"]:
    print(f"\n=== {area} ===")
    cols = np.where(brain_areas_str == area)[0]
    tensor = _build_trial_tensor(
        responses_pp, trial_boundaries, clip_trial_indices, cols,
        N_FRAMES_TRUNCATE,
    )
    print(f"  trial tensor shape: {tensor.shape}")
    pca_a = pca_per_area[area]['pca']
    stim_pcs_a = pca_per_area[area]['stim_pcs']
    d_full = traj_utils.pairwise_trajectory_distance(
        trajectories[area], metric="full",
    )
    d_top3 = traj_utils.pairwise_trajectory_distance(
        trajectories[area], metric="top_pcs",
        pca=pca_a, n_pcs=N_PCS_FOR_DISTANCE,
    )
    print(f"  bootstrap envelopes (B={N_BOOT})...")
    env_full = traj_utils.bootstrap_distance_envelope(
        tensor, clip_category_labels, metric="full",
        n_boot=N_BOOT, seed=RANDOM_SEED + 10 + ord(area[0]),
    )
    env_top3 = traj_utils.bootstrap_distance_envelope(
        tensor, clip_category_labels, metric="top_pcs",
        pca=pca_a, n_pcs=N_PCS_FOR_DISTANCE,
        n_boot=N_BOOT, seed=RANDOM_SEED + 20 + ord(area[0]),
    )
    print(f"  shuffle null (n={N_SHUFFLES})...")
    null_full = traj_utils.shuffle_null_max_distance(
        tensor, clip_category_labels, metric="full",
        n_shuffles=N_SHUFFLES, seed=RANDOM_SEED + 30 + ord(area[0]),
    )
    null_top3 = traj_utils.shuffle_null_max_distance(
        tensor, clip_category_labels, metric="top_pcs",
        pca=pca_a, n_pcs=N_PCS_FOR_DISTANCE,
        n_shuffles=N_SHUFFLES, seed=RANDOM_SEED + 40 + ord(area[0]),
    )
    all_area_results[area] = {
        "tensor": tensor,
        "trajectories": trajectories[area],
        "pca": pca_a,
        "stim_pcs": stim_pcs_a,
        "distances_full": d_full,
        "distances_top3": d_top3,
        "envelopes_full": env_full,
        "envelopes_top3": env_top3,
        "null_full": null_full,
        "null_top3": null_top3,
        "n_neurons": tensor.shape[2],
    }

# Per-area scatters.
for area in ["AL", "LM", "RL"]:
    res = all_area_results[area]
    fig, ax = plt.subplots(figsize=(8, 6))
    traj_utils.plot_trajectory_2d(
        res['stim_pcs'], area, res['pca'],
        palette=o4.CATEGORY_COLORS, stim_order=o4.CATEGORY_ORDER,
        ax=ax,
    )
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / f"2_3_{area.lower()}_trajectory_2d.png", dpi=150)
    plt.show()
    fig3d = traj_utils.plot_trajectory_3d_plotly(
        res['stim_pcs'], area, res['pca'],
        palette=o4.CATEGORY_COLORS, stim_order=o4.CATEGORY_ORDER,
    )
    fig3d.write_html(str(FIGURES_DIR / f"2_3_{area.lower()}_trajectory_3d.html"))